# 🧮 Personal Budget Forecasting (Kaggle Style)
Regression with SHAP + Gradio for monthly spending prediction.

In [ ]:
!pip install lightgbm shap gradio --quiet

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import shap
import gradio as gr


In [ ]:
np.random.seed(333)
n = 1400
df = pd.DataFrame({
    "income": np.random.randint(2000, 10000, n),
    "rent": np.random.randint(500, 3000, n),
    "family_size": np.random.randint(1, 6, n),
    "debt_payments": np.random.randint(0, 1000, n),
    "spending": np.random.randint(1000, 9000, n)
})


In [ ]:
X = df.drop(columns="spending")
y = df["spending"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=333)
model = lgb.LGBMRegressor()
model.fit(X_train, y_train)
print("RMSE:", np.sqrt(mean_squared_error(y_test, model.predict(X_test))))


In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
def predict_budget(income, rent, family_size, debt_payments):
    row = pd.DataFrame([[income, rent, family_size, debt_payments]], columns=X.columns)
    pred = model.predict(row)[0]
    return f"Estimated Monthly Spending: ${pred:,.0f}"

gr.Interface(
    fn=predict_budget,
    inputs=[
        gr.Number(label="Monthly Income ($)"),
        gr.Number(label="Rent/Mortgage ($)"),
        gr.Slider(1, 6, step=1, label="Family Size"),
        gr.Number(label="Debt Payments ($)")
    ],
    outputs="text",
    title="Monthly Budget Estimator"
).launch()
